# Context-aware chat MT — English↔Chinese pipeline on Kaggle

Runs `scripts/run_zh_pipeline.sh` (greedy no/full-context translation → evaluation → epsilon-sampling candidates →
COMET / context-COMET MBR → contrastive decoding → P-CXMI → report) with **TowerInstruct-7B-v0.2** and **COMET-22**.

**Before running** (right-hand panel → *Session options*):
1. **Accelerator: GPU T4 x2** (the 7B model is split over both T4s; T4s have no bfloat16, so float16 is used).
2. **Internet: On** (needs a phone-verified Kaggle account) — models are downloaded from HuggingFace.
3. Private GitHub repo? Add a Kaggle secret named `GITHUB_TOKEN` (Add-ons → Secrets) with a read-only token.

Start with the small run (cell 5) before launching bigger ones.

In [ ]:
# 1. GPUs
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Get the code + en<->zh data only (the full repo is ~4.4 GB; sparse checkout keeps it small)
import os, subprocess
REPO = "namannamu18/context-aware-translation"
BRANCH = "master"           # change if the Kaggle changes are still on another branch
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass                    # public repo: no token needed
url = f"https://{token+'@' if token else ''}github.com/{REPO}.git"
os.chdir("/kaggle/working")
if not os.path.exists("cat"):
    subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", "-b", BRANCH, url, "cat"], check=True)
os.chdir("/kaggle/working/cat")
dirs = ["scripts", "configs/zh", "paper_results_zh"] + [
    f"raw_data/mt/{ds}.{x}" for ds in ["bmeld_test", "bmeld_dev", "bmeld_train", "synthetic_chat_test"] for x in ["zh", "en-zh", "zh-en"]]
subprocess.run(["git", "sparse-checkout", "set"] + dirs, check=True)
!ls && ls raw_data/mt

In [ ]:
# 3. Install (vLLM 0.6.x = the version tower-eval/the original scripts were written for; supports T4)
!pip install -q "vllm==0.6.4.post1" "transformers==4.46.3" "unbabel-comet==2.2.7" "sacrebleu>=2.4" "numpy<2" "protobuf<5" pandas pyyaml
# If later cells fail with torch/CUDA import errors: Run -> Restart session, then continue from cell 4.

In [ ]:
# 4. Settings shared by all runs
import os
os.chdir("/kaggle/working/cat")
os.environ.update({
    "PYTHON": "python",
    "MODEL": "Unbabel/TowerInstruct-7B-v0.2",
    "MODEL_NAME": "TowerInstruct-7B-v0.2",
    "BACKEND": "vllm",
    "SEP_TOKENS": "tower",                      # Tower token ids for P-CXMI gating (as in the paper)
    "COMET_MODEL": "Unbabel/wmt22-comet-da",
    "UTILITY": "comet",
    # T4: no bfloat16 -> float16; split the 7B model over the two T4s.
    # Lower memory use + small batches leave room for P-CXMI's full-vocabulary prompt log-probs (OOM otherwise)
    "VLLM_ENGINE_ARGS": '{"dtype": "half", "tensor_parallel_size": 2, "max_model_len": 4096, "gpu_memory_utilization": 0.75, "max_num_seqs": 8, "max_num_batched_tokens": 4096}',
    "CD_EXTRA_ARGS": "--torch_dtype float16 --device_map auto",
    "TOKENIZERS_PARALLELISM": "false",
    # 1.0 = paper setup; 1.1 stops rare repetition loops ("啊，啊，啊…") in greedy translation
    "REPETITION_PENALTY": "1.1",
    # Kaggle preinstalls TensorFlow, which breaks with protobuf<5 (needed by COMET); it is not used here
    "USE_TF": "0",
    "TRANSFORMERS_NO_TF": "1",
})

In [ ]:
# 5. SMALL test run: 5 BMELD-test conversations, 8 candidates (expect roughly 30-60 min, mostly model loading
#    and contrastive decoding). Check that it finishes before starting larger runs.
!ROOT=/kaggle/working/run_small DATA_NAME=bmeld SPLIT=test MAX_DOCS=5 N_CANDIDATES=8 MAX_TOKENS=256 \
    CONTEXT_SIZES="0 2" bash scripts/run_zh_pipeline.sh 2>&1 | grep -v -E "it/s\]|s/it\]" | tail -40

In [ ]:
# 6. Consistency checks (prompt alignment, P-CXMI token gating, MBR selections, line counts)
!python scripts/check_zh_pipeline_outputs.py --root_dir /kaggle/working/run_small --data_name bmeld --split test \
    --model Unbabel/TowerInstruct-7B-v0.2 --model_name TowerInstruct-7B-v0.2 --gen_backend vllm --n_candidates 8 2>/dev/null | tail -15

In [ ]:
# 7. Results of the run (greedy no/full context, MBR variants, contrastive decoding, P-CXMI)
import json
rep = json.load(open("/kaggle/working/run_small/pipeline_report.json"))
for k, v in rep["greedy"].items(): print("greedy", k, v)
for k, v in rep["mbr"].items(): print("MBR", k, {lp: d["mbr"] for lp, d in v.items()})
for k, v in rep["contrastive"].items(): print("contrastive", k, v)
print(json.dumps(rep["pcxmi"], indent=1)[:2000])

## Bigger runs

Scale up step by step (each line is one run; outputs go to separate folders). The paper setting is
`N_CANDIDATES=100` and `CONTEXT_SIZES="0 2 6 10 15"` on the full test set (2,601 segments) — that is many GPU hours
and will not fit in one 12-hour Kaggle session, so split it (e.g. `SKIP_DOCS=0 MAX_DOCS=70`, `SKIP_DOCS=70 MAX_DOCS=70`, …).

For long runs use **Save Version → Save & Run All (Commit)**: it runs in the background (up to 12 h) and keeps
everything written to `/kaggle/working` as the notebook's output.

In [ ]:
# 8. Example: 50 conversations of BMELD test with 32 candidates
!ROOT=/kaggle/working/run_bmeld_test_50 DATA_NAME=bmeld SPLIT=test MAX_DOCS=50 N_CANDIDATES=32 MAX_TOKENS=256 \
    CONTEXT_SIZES="0 2 6" bash scripts/run_zh_pipeline.sh 2>&1 | grep -v -E "it/s\]|s/it\]" | tail -20

In [ ]:
# 9. Download results (candidates are large, so they are left out of the zip)
!cd /kaggle/working && zip -qr results.zip run_* -x "*/candidates/*" && ls -lh results.zip